# `config.py`

## Structure summary

```text
config.py
│
├── Dataclasses (parameter containers)
│   │
│   ├── TaskConfig                              L15–L23
│   │       └── n_lists, items_per_list, items_per_train,
│   │           reward_mean, reward_std, initial_value
│   │
│   ├── ModelConfig                             L29–L38
│   │       └── alpha, beta_base, delta_boundary,
│   │           rpe_drift_weight, recall_intercept,
│   │           recall_drift_weight, recall_noise_std
│   │
│   ├── HypothesisParams                        L44–L56
│   │       └── h1_delta_boundary, h2_bias_shift
│   │
│   └── SimConfig                               L62–L67
│           └── n_subjects, seed, hypothesis_names
│
└── Convenience accessors
    │
    ├── get_default_task_config()                L73–L74
    ├── get_default_model_config()               L77–L78
    ├── get_hypothesis_params()                  L81–L82
    ├── get_default_sim_config()                 L85–L86
    └── get_all_configs()                        L89–L96
            └── returns dict of all four default config objects
```

## Function summary

| Function | Purpose | Output |
|---|---|---|
| `get_default_task_config` | Instantiate `TaskConfig` with defaults | `TaskConfig` dataclass |
| `get_default_model_config` | Instantiate `ModelConfig` with defaults | `ModelConfig` dataclass |
| `get_hypothesis_params` | Instantiate `HypothesisParams` with defaults | `HypothesisParams` dataclass |
| `get_default_sim_config` | Instantiate `SimConfig` with defaults | `SimConfig` dataclass |
| `get_all_configs` | Return a dict of all four default config objects | `dict` with keys `task`, `model`, `hypothesis`, `sim` |

# `config.py` pseudocodes

## `TaskConfig`

**Implementation span:** `L15–L23`

### Pseudocode
- Define a frozen set of task-geometry constants: number of study lists, items per list, items per train (segment), reward distribution parameters, and the starting expected value.

| Field | Default | Role |
|---|---|---|
| `n_lists` | `4` | number of study lists |
| `items_per_list` | `24` | items presented per list |
| `items_per_train` | `6` | items between successive boundaries |
| `reward_mean` | `50.0` | centre of reward sampling distribution |
| `reward_std` | `20.0` | spread of reward sampling distribution |
| `initial_value` | `50.0` | starting expected value estimate |

## `ModelConfig`

**Implementation span:** `L29–L38`

### Pseudocode
- Define parameters shared across all hypotheses: RL learning rate, baseline encoding drift, boundary drift increment, RPE-to-drift coupling, and the logistic recall link function parameters.

| Field | Default | Role |
|---|---|---|
| `alpha` | `0.3` | Rescorla–Wagner learning rate |
| `beta_base` | `0.5` | baseline encoding drift |
| `delta_boundary` | `0.25` | boundary drift increment (H0 & H2) |
| `rpe_drift_weight` | `0.15` | \|RPE\| → drift coupling strength |
| `recall_intercept` | `-0.5` | logistic recall intercept |
| `recall_drift_weight` | `2.0` | drift → recall logistic weight |
| `recall_noise_std` | `0.05` | Gaussian noise on recall logit |

## `HypothesisParams`

**Implementation span:** `L44–L56`

### Pseudocode
- Define the values that distinguish each alternative hypothesis from baseline: a reduced boundary increment for H1, and a global bias shift for H2.

| Field | Default | Role |
|---|---|---|
| `h1_delta_boundary` | `0.10` | smaller boundary increment used by H1 |
| `h2_bias_shift` | `0.15` | global downward shift applied by H2 |

## `SimConfig`

**Implementation span:** `L62–L67`

### Pseudocode
- Define top-level run settings: number of simulated subjects, RNG seed, and the ordered tuple of hypothesis names to simulate.

| Field | Default | Role |
|---|---|---|
| `n_subjects` | `100` | simulated subjects per hypothesis |
| `seed` | `42` | master RNG seed for reproducibility |
| `hypothesis_names` | `("baseline", "H1_smaller_boundary", "H2_global_reduction")` | hypotheses to iterate over |

## `get_all_configs`

**Implementation span:** `L89–L96`

### Pseudocode
- Instantiate each of the four config dataclasses with their defaults and return them in a single dictionary.

| Pseudocode step | Lines | Code |
|---|---|---|
| Build and return dict of all defaults | `L91–L96` | `return {"task": …, "model": …, "hypothesis": …, "sim": …}` |

# `utils.py`

## Flowchart

```text
utils.py  —  standalone helper functions, no cross-dependencies
│
├── set_seed(seed)                              L12–L14
│       └── returns np.random.Generator
│
├── sigmoid(x)                                  L17–L24
│       └── numerically stable σ(x), scalar or array
│
├── ensure_dir(path)                            L27–L31
│       └── mkdir -p, returns Path
│
├── save_dataframe(df, path)                    L34–L39
│       └── ensure parent dir → df.to_csv
│
├── save_json(obj, path)                        L42–L48
│       └── ensure parent dir → json.dump
│
└── zscore_safe(x)                              L51–L57
        └── z-score that returns zeros when std == 0
```

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `set_seed` | Create a seeded NumPy Generator | `seed`: int | `np.random.Generator` |
| `sigmoid` | Numerically stable sigmoid | `x`: scalar or array | same shape, values in (0, 1) |
| `ensure_dir` | Create directory tree if needed | `path`: str or Path | `Path` object |
| `save_dataframe` | Save a DataFrame to CSV | `df`, `path` | side-effect: writes CSV |
| `save_json` | Serialise object to JSON | `obj`, `path` | side-effect: writes JSON |
| `zscore_safe` | Z-score that handles zero variance | `x`: array-like | array of z-scores or zeros |

# `utils.py` pseudocodes

## `set_seed`

**Implementation span:** `L12–L14`

### Pseudocode
- Construct and return a NumPy random Generator from the given integer seed.

| Pseudocode step | Lines | Code |
|---|---|---|
| Create generator | `L14` | `return np.random.default_rng(seed)` |

## `sigmoid`

**Implementation span:** `L17–L24`

### Pseudocode
- Cast input to a float array.
- For non-negative values, compute `1 / (1 + exp(-x))`.
- For negative values, compute the numerically equivalent `exp(x) / (1 + exp(x))` to avoid overflow.

| Pseudocode step | Lines | Code |
|---|---|---|
| Cast to float array | `L19` | `x = np.asarray(x, dtype=float)` |
| Branch on sign for stability | `L20–L24` | `return np.where(x >= 0, 1/(1+exp(-x)), exp(x)/(1+exp(x)))` |

## `ensure_dir`

**Implementation span:** `L27–L31`

### Pseudocode
- Convert the path to a `Path` object.
- Create the directory and any missing parents.
- Return the `Path`.

| Pseudocode step | Lines | Code |
|---|---|---|
| Convert to Path | `L29` | `p = Path(path)` |
| Create directory tree | `L30` | `p.mkdir(parents=True, exist_ok=True)` |
| Return Path | `L31` | `return p` |

## `save_dataframe`

**Implementation span:** `L34–L39`

### Pseudocode
- Convert path to a `Path` object.
- Ensure the parent directory exists.
- Write the DataFrame to CSV without the index.
- Print a confirmation with the row count.

| Pseudocode step | Lines | Code |
|---|---|---|
| Ensure parent dir | `L36–L37` | `p = Path(path); ensure_dir(p.parent)` |
| Write CSV | `L38` | `df.to_csv(p, index=False)` |
| Print confirmation | `L39` | `print(f"  Saved {len(df)} rows → {p}")` |

## `save_json`

**Implementation span:** `L42–L48`

### Pseudocode
- Convert path to a `Path` object.
- Ensure the parent directory exists.
- Open the file for writing and dump the object as indented JSON.
- Print a confirmation.

| Pseudocode step | Lines | Code |
|---|---|---|
| Ensure parent dir | `L44–L45` | `p = Path(path); ensure_dir(p.parent)` |
| Write JSON | `L46–L47` | `json.dump(obj, f, indent=2, default=str)` |
| Print confirmation | `L48` | `print(f"  Saved JSON → {p}")` |

## `zscore_safe`

**Implementation span:** `L51–L57`

### Pseudocode
- Cast input to float array.
- Compute the sample standard deviation.
- If the standard deviation is zero, return an array of zeros to avoid division by zero.
- Otherwise return the standard z-score.

| Pseudocode step | Lines | Code |
|---|---|---|
| Cast to float array | `L53` | `x = np.asarray(x, dtype=float)` |
| Compute sample SD | `L54` | `s = np.std(x, ddof=1)` |
| Guard against zero variance | `L55–L56` | `if s == 0: return np.zeros_like(x)` |
| Return z-scores | `L57` | `return (x - np.mean(x)) / s` |

# `task_design.py`

## Flowchart

```text
task_design.py
│
├── make_exp1_schedule(task_cfg, rng)                        L21–L84
│       │
│       ├── for each list:
│       │       for each within-list position:
│       │           ├── flag boundary items (every items_per_train positions)
│       │           ├── increment train_id at each boundary
│       │           ├── sample reward from N(reward_mean, reward_std), clipped [0, 100]
│       │           └── append row dict to list
│       │
│       └── return pd.DataFrame
│
├── assign_boundary_flags(df)                                L87–L93
│       └── stub / pass-through (flags already set in make_exp1_schedule)
│
├── assign_rpe_values(df, initial_value, alpha)              L96–L123
│       │
│       ├── initialize expected value to initial_value
│       ├── for each trial:
│       │       ├── signed_rpe = reward − expected_value
│       │       └── if next trial is a new list: reset EV
│       │           else: RW update EV
│       │
│       └── return df with expected_value, outcome_rpe, abs_outcome_rpe
│
└── build_trial_dataframe(task_cfg, rng, alpha)              L126–L134
        │
        ├── call make_exp1_schedule(task_cfg, rng)
        ├── call assign_rpe_values(df, initial_value, alpha)
        └── return annotated DataFrame
```

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `make_exp1_schedule` | Build the full trial-level schedule for one subject | `task_cfg`, optional `rng` | DataFrame with columns: trial, list_id, item_id, within_list_pos, train_id, within_train_pos, is_boundary, reward |
| `assign_boundary_flags` | Stub for recomputing boundary labels on a loaded schedule | `df` | DataFrame (unchanged) |
| `assign_rpe_values` | Compute expected value, signed RPE, and \|RPE\| via a Rescorla–Wagner rule | `df`, `initial_value`, `alpha` | DataFrame with added columns: expected_value, outcome_rpe, abs_outcome_rpe |
| `build_trial_dataframe` | One-call convenience: schedule → RPE annotation | `task_cfg`, optional `rng`, `alpha` | Fully annotated trial DataFrame |

# `task_design.py` pseudocodes

## `make_exp1_schedule`

**Implementation span:** `L21–L84`

### Pseudocode
- Default to a fresh RNG if none is provided.
- Initialize an empty row list, a global trial counter, and a global train counter.
- For each list, reset within-train position and iterate over items per list.
- At every `items_per_train`-th position (after the first), flag a boundary and increment the train counter.
- Sample a reward from a clipped Gaussian.
- Append a row dictionary with trial, list_id, item_id, within_list_pos, train_id, within_train_pos, is_boundary, and reward.
- After finishing a list, increment the train counter again (new list = new train).
- Return the rows as a DataFrame.

| Pseudocode step | Lines | Code |
|---|---|---|
| Default RNG | `L40–L41` | `if rng is None: rng = np.random.default_rng()` |
| Initialize counters | `L43–L45` | `rows = []; trial = 0; global_train = 0` |
| Outer loop over lists | `L47` | `for list_id in range(task_cfg.n_lists):` |
| Inner loop over positions | `L49` | `for pos in range(task_cfg.items_per_list):` |
| Flag boundary | `L50–L52` | `is_boundary = int(pos > 0 and pos % task_cfg.items_per_train == 0)` |
| Increment train at boundary | `L53–L55` | `if is_boundary: global_train += 1; within_train_pos = 0` |
| Sample clipped reward | `L57–L63` | `reward = float(np.clip(rng.normal(…), 0, 100))` |
| Append row dict | `L65–L76` | `rows.append({…})` |
| New list increments train | `L81` | `global_train += 1` |
| Return DataFrame | `L83–L84` | `df = pd.DataFrame(rows); return df` |

## `assign_rpe_values`

**Implementation span:** `L96–L123`

### Pseudocode
- Allocate expected-value and signed-RPE arrays of length `n`.
- Set the first expected value to `initial_value`.
- For each trial, compute the signed RPE as `reward − EV`.
- If the next trial starts a new list (within_list_pos == 0), reset EV; otherwise apply the Rescorla–Wagner update.
- Copy the DataFrame and attach expected_value, outcome_rpe, and abs_outcome_rpe columns.

| Pseudocode step | Lines | Code |
|---|---|---|
| Allocate arrays | `L105–L107` | `n = len(df); ev = np.zeros(n); signed_rpe = np.zeros(n)` |
| Set initial EV | `L109` | `ev[0] = initial_value` |
| Loop over trials | `L110` | `for t in range(n):` |
| Compute signed RPE | `L111` | `signed_rpe[t] = df.iloc[t]["reward"] - ev[t]` |
| Reset or update EV | `L112–L117` | `if df.iloc[t+1]["within_list_pos"] == 0: ev[t+1] = initial_value else: ev[t+1] = ev[t] + alpha * signed_rpe[t]` |
| Attach columns to copy | `L119–L123` | `df = df.copy(); df["expected_value"] = …; df["outcome_rpe"] = …; df["abs_outcome_rpe"] = …` |

## `build_trial_dataframe`

**Implementation span:** `L126–L134`

### Pseudocode
- Call `make_exp1_schedule` to generate the trial structure.
- Call `assign_rpe_values` to annotate with RPE columns.
- Return the fully annotated DataFrame.

| Pseudocode step | Lines | Code |
|---|---|---|
| Generate schedule | `L132` | `df = make_exp1_schedule(task_cfg, rng=rng)` |
| Annotate with RPE | `L133` | `df = assign_rpe_values(df, initial_value=task_cfg.initial_value, alpha=alpha)` |
| Return | `L134` | `return df` |

# `hypotheses.py`

## Flowchart

```text
hypotheses.py
│
├── Per-trial drift rules (private)
│   │
│   └── _base_drift(is_boundary, abs_rpe, beta_base, delta_boundary, rpe_weight)   L31–L40
│           └── drift = beta_base + delta_boundary * is_boundary + rpe_weight * (abs_rpe / 100)
│
├── Hypothesis-specific wrappers
│   │
│   ├── baseline_drift_rule(is_boundary, abs_rpe, model_cfg, _hyp)                 L43–L56
│   │       └── calls _base_drift with model_cfg.beta_base, model_cfg.delta_boundary
│   │
│   ├── smaller_boundary_increase_rule(is_boundary, abs_rpe, model_cfg, hyp)        L59–L72
│   │       └── calls _base_drift with model_cfg.beta_base, hyp.h1_delta_boundary
│   │
│   └── global_bias_reduction_rule(is_boundary, abs_rpe, model_cfg, hyp)            L75–L89
│           └── shifted_base = model_cfg.beta_base − hyp.h2_bias_shift
│               calls _base_drift with shifted_base, model_cfg.delta_boundary
│
├── Dispatch table
│   │
│   └── _RULES = {"baseline": …, "H1_smaller_boundary": …, "H2_global_reduction": …}
│
├── compute_encoding_drift(trial_df, hypothesis_name, model_cfg, hyp)               L103–L137
│       │
│       ├── look up rule function from _RULES
│       ├── for each row in trial_df:
│       │       call rule(is_boundary, abs_outcome_rpe, model_cfg, hyp)
│       └── return array of per-trial drift values
│
└── list_hypotheses()                                                                L140–L142
        └── return list of keys from _RULES
```

## Hypothesis parameter comparison

| Parameter | Baseline (H0) | H1 Smaller Δ | H2 Global ↓ |
|---|---|---|---|
| non-boundary drift | `beta_base` | `beta_base` | `beta_base − h2_bias_shift` |
| boundary drift | `beta_base + delta_boundary` | `beta_base + h1_delta_boundary` | `(beta_base − h2_bias_shift) + delta_boundary` |
| RPE modulation | `rpe_drift_weight × (\|RPE\| / 100)` | same | same |

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `_base_drift` | Core drift formula shared by all hypotheses | `is_boundary`, `abs_rpe`, `beta_base`, `delta_boundary`, `rpe_weight` | `float` drift value |
| `baseline_drift_rule` | H0 wrapper: default parameters | `is_boundary`, `abs_rpe`, `model_cfg` | `float` drift |
| `smaller_boundary_increase_rule` | H1 wrapper: smaller boundary increment | `is_boundary`, `abs_rpe`, `model_cfg`, `hyp` | `float` drift |
| `global_bias_reduction_rule` | H2 wrapper: shifted baseline | `is_boundary`, `abs_rpe`, `model_cfg`, `hyp` | `float` drift |
| `compute_encoding_drift` | Vectorised dispatch over all trials | `trial_df`, `hypothesis_name`, `model_cfg`, `hyp` | `(n_trials,)` float array |
| `list_hypotheses` | Return available hypothesis names | — | `list[str]` |

# `hypotheses.py` pseudocodes

## `_base_drift`

**Implementation span:** `L31–L40`

### Pseudocode
- Compute encoding drift as the sum of: baseline, boundary increment scaled by boundary flag, and RPE modulation scaled by \|RPE\| / 100.
- Return the scalar drift value.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute drift | `L39` | `drift = beta_base + delta_boundary * is_boundary + rpe_weight * (abs_rpe / 100.0)` |
| Return | `L40` | `return drift` |

## `baseline_drift_rule`

**Implementation span:** `L43–L56`

### Pseudocode
- Call `_base_drift` using the standard model config values: `beta_base`, `delta_boundary`, and `rpe_drift_weight`.

| Pseudocode step | Lines | Code |
|---|---|---|
| Delegate to core formula | `L50–L56` | `return _base_drift(is_boundary, abs_rpe, model_cfg.beta_base, model_cfg.delta_boundary, model_cfg.rpe_drift_weight)` |

## `smaller_boundary_increase_rule`

**Implementation span:** `L59–L72`

### Pseudocode
- Call `_base_drift` with the standard `beta_base` but substitute `hyp.h1_delta_boundary` for the boundary increment.

| Pseudocode step | Lines | Code |
|---|---|---|
| Delegate with smaller delta | `L66–L72` | `return _base_drift(is_boundary, abs_rpe, model_cfg.beta_base, hyp.h1_delta_boundary, model_cfg.rpe_drift_weight)` |

## `global_bias_reduction_rule`

**Implementation span:** `L75–L89`

### Pseudocode
- Compute a shifted baseline as `beta_base − h2_bias_shift`.
- Call `_base_drift` with the shifted baseline and the standard `delta_boundary`.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute shifted baseline | `L82` | `shifted_base = model_cfg.beta_base - hyp.h2_bias_shift` |
| Delegate with shifted base | `L83–L89` | `return _base_drift(is_boundary, abs_rpe, shifted_base, model_cfg.delta_boundary, model_cfg.rpe_drift_weight)` |

## `compute_encoding_drift`

**Implementation span:** `L103–L137`

### Pseudocode
- Look up the appropriate rule function from the `_RULES` dispatch table using the hypothesis name.
- Iterate over every row in the trial DataFrame.
- For each row, call the rule with `is_boundary`, `abs_outcome_rpe`, `model_cfg`, and `hyp`.
- Collect results into a NumPy array and return it.

| Pseudocode step | Lines | Code |
|---|---|---|
| Look up rule | `L125` | `rule = _RULES[hypothesis_name]` |
| Iterate over rows and apply rule | `L126–L136` | `drifts = np.array([rule(int(row["is_boundary"]), float(row["abs_outcome_rpe"]), model_cfg, hyp) for _, row in trial_df.iterrows()])` |
| Return drift array | `L137` | `return drifts` |

## `list_hypotheses`

**Implementation span:** `L140–L142`

### Pseudocode
- Return the list of keys from the `_RULES` dispatch dictionary.

| Pseudocode step | Lines | Code |
|---|---|---|
| Return keys | `L142` | `return list(_RULES.keys())` |

# `simulator.py`

## Flowchart

```text
simulator.py
│
├── RL helpers
│   ├── compute_outcome_rpe(reward, expected_value)              L25–L27
│   │       └── return reward − expected_value
│   │
│   └── update_expected_value(prev_value, reward, alpha)         L30–L34
│           └── return prev_value + alpha * (reward − prev_value)
│
├── simulate_subject(trial_df, hypothesis_name, model_cfg, hyp, rng)    L41–L82
│       │
│       ├── copy trial_df
│       ├── call compute_encoding_drift(df, hypothesis_name, …)
│       │       └── attach encoding_drift column
│       │
│       ├── compute logit = recall_intercept + recall_drift_weight × drift
│       ├── add Gaussian noise → sigmoid → recall_prob
│       ├── draw binary recalled from Bernoulli(recall_prob)
│       └── return df with encoding_drift, recall_prob, recalled, hypothesis
│
├── simulate_group(task_cfg, hypothesis_name, model_cfg, hyp, n_subjects, rng_seed)   L89–L122
│       │
│       ├── create master RNG from seed
│       ├── for each subject:
│       │       ├── spawn subject RNG → build_trial_dataframe (independent rewards)
│       │       ├── spawn recall RNG  → simulate_subject
│       │       └── tag subject_id
│       │
│       └── concatenate all subject DataFrames and return
│
└── run_all_hypotheses(task_cfg, model_cfg, hyp, sim_cfg)                L129–L163
        │
        ├── fill in defaults for any None config
        ├── for each hypothesis in sim_cfg.hypothesis_names:
        │       call simulate_group(…)
        │
        └── concatenate across hypotheses and return long-form DataFrame
```

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `compute_outcome_rpe` | Signed prediction error | `reward`, `expected_value` | `float` |
| `update_expected_value` | Rescorla–Wagner value update | `prev_value`, `reward`, `alpha` | `float` |
| `simulate_subject` | Run one simulated subject under one hypothesis | `trial_df`, `hypothesis_name`, `model_cfg`, `hyp`, `rng` | DataFrame with `encoding_drift`, `recall_prob`, `recalled`, `hypothesis` |
| `simulate_group` | Simulate `n_subjects` under one hypothesis | `task_cfg`, `hypothesis_name`, `model_cfg`, `hyp`, `n_subjects`, `rng_seed` | Long-form DataFrame with `subject_id` |
| `run_all_hypotheses` | Simulate all hypotheses and concatenate | optional config overrides | Single long-form DataFrame |

# `simulator.py` pseudocodes

## `compute_outcome_rpe`

**Implementation span:** `L25–L27`

### Pseudocode
- Return the signed difference between the reward received and the current expected value.

| Pseudocode step | Lines | Code |
|---|---|---|
| Compute and return RPE | `L27` | `return reward - expected_value` |

## `update_expected_value`

**Implementation span:** `L30–L34`

### Pseudocode
- Apply the Rescorla–Wagner delta rule: shift the previous value toward the reward by a fraction `alpha`.

| Pseudocode step | Lines | Code |
|---|---|---|
| RW update | `L34` | `return prev_value + alpha * (reward - prev_value)` |

## `simulate_subject`

**Implementation span:** `L41–L82`

### Pseudocode
- Copy the trial schedule so the original is not mutated.
- Call `compute_encoding_drift` to get the hypothesis-specific drift vector and attach it to the DataFrame.
- Compute the recall logit as `recall_intercept + recall_drift_weight × drift`.
- Add Gaussian noise and pass through the sigmoid to get recall probabilities.
- Draw binary recall outcomes from Bernoulli(recall_prob).
- Tag the hypothesis name and return the augmented DataFrame.

| Pseudocode step | Lines | Code |
|---|---|---|
| Copy schedule | `L66` | `df = trial_df.copy()` |
| Compute encoding drift | `L69–L70` | `drift = compute_encoding_drift(df, hypothesis_name, model_cfg, hyp); df["encoding_drift"] = drift` |
| Compute logit | `L73` | `logit = model_cfg.recall_intercept + model_cfg.recall_drift_weight * drift` |
| Add noise and sigmoid | `L74–L76` | `noise = rng.normal(…); recall_prob = sigmoid(logit + noise); df["recall_prob"] = …` |
| Draw binary recall | `L79` | `df["recalled"] = (rng.random(len(df)) < recall_prob).astype(int)` |
| Tag hypothesis and return | `L81–L82` | `df["hypothesis"] = hypothesis_name; return df` |

## `simulate_group`

**Implementation span:** `L89–L122`

### Pseudocode
- Create a master RNG from the seed to ensure reproducibility.
- For each subject, spawn two child RNGs: one for reward sampling, one for recall noise.
- Build a fresh trial schedule (with independent reward draws) via `build_trial_dataframe`.
- Simulate the subject via `simulate_subject` using the recall RNG.
- Tag the subject ID and collect the DataFrame.
- Concatenate all subject DataFrames and return.

| Pseudocode step | Lines | Code |
|---|---|---|
| Create master RNG | `L106` | `master_rng = np.random.default_rng(rng_seed)` |
| Loop over subjects | `L109` | `for s in range(n_subjects):` |
| Spawn subject RNG and build schedule | `L111–L112` | `subj_rng = np.random.default_rng(master_rng.integers(…)); trial_df = build_trial_dataframe(…)` |
| Spawn recall RNG and simulate | `L115–L118` | `recall_rng = …; subj_df = simulate_subject(trial_df, hypothesis_name, model_cfg, hyp, recall_rng)` |
| Tag subject_id | `L119` | `subj_df["subject_id"] = s` |
| Concatenate and return | `L122` | `return pd.concat(frames, ignore_index=True)` |

## `run_all_hypotheses`

**Implementation span:** `L129–L163`

### Pseudocode
- Import default config constructors and fill in any `None` arguments.
- For each hypothesis name in the simulation config, call `simulate_group` and collect the resulting DataFrame.
- Concatenate all hypothesis DataFrames into one long-form DataFrame and return it.

| Pseudocode step | Lines | Code |
|---|---|---|
| Fill defaults | `L139–L149` | `task_cfg = task_cfg or get_default_task_config(); …` |
| Loop over hypotheses | `L152` | `for h_name in sim_cfg.hypothesis_names:` |
| Simulate group | `L154–L158` | `df = simulate_group(task_cfg, h_name, model_cfg, hyp, n_subjects=…, rng_seed=…)` |
| Concatenate and return | `L161–L163` | `all_df = pd.concat(frames, ignore_index=True); return all_df` |

# `metrics.py`

## Flowchart

```text
trial_df:  long-form DataFrame (trial × subject × hypothesis)
│
├── Trial → Subject aggregation
│   │
│   ├── compute_recall_accuracy(df)                          L20–L27
│   │       └── mean recalled per subject × hypothesis
│   │
│   ├── compute_boundary_vs_nonboundary_recall(df)           L30–L47
│   │       ├── group by hypothesis, subject_id, is_boundary
│   │       ├── pivot to wide: nonboundary_recall, boundary_recall
│   │       └── compute boundary_advantage = boundary − nonboundary
│   │
│   ├── compute_serial_position_curve(df)                    L50–L57
│   │       └── mean recall at each within_list_pos per hypothesis
│   │
│   ├── compute_recall_by_train(df)                          L60–L67
│   │       └── mean recall per train per subject × hypothesis
│   │
│   ├── compute_drift_summary(df)                            L70–L90
│   │       ├── group by hypothesis, subject_id, is_boundary
│   │       ├── pivot to wide: mean_nonboundary_drift, mean_boundary_drift
│   │       └── compute drift_boundary_contrast
│   │
│   └── compute_rpe_recall_coupling(df)                      L93–L105
│           └── per-subject correlation(|RPE|, recalled)
│
├── Subject-level summary
│   │
│   └── compute_subject_summary(df)                          L112–L138
│           ├── call each Trial → Subject function above
│           ├── compute mean_abs_rpe and mean_drift per subject
│           └── left-merge all into one wide DataFrame
│
├── Hypothesis-level aggregate
│   │
│   └── compute_hypothesis_summary(subject_df)               L145–L154
│           ├── identify numeric columns (excluding subject_id)
│           ├── group by hypothesis, aggregate mean and SEM
│           └── flatten multi-level column names
│
└── All-in-one
    │
    └── summarize_all_metrics(trial_df)                      L161–L167
            ├── call compute_subject_summary
            ├── call compute_hypothesis_summary
            ├── call compute_serial_position_curve
            ├── call compute_recall_by_train
            └── return (subject_summary, hypothesis_summary,
                        serial_position, train_recall)
```

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `compute_recall_accuracy` | Overall recall proportion per subject × hypothesis | trial-level `df` | DataFrame: hypothesis, subject_id, overall_recall |
| `compute_boundary_vs_nonboundary_recall` | Recall split by boundary flag; compute boundary advantage | trial-level `df` | Wide DataFrame: nonboundary_recall, boundary_recall, boundary_advantage |
| `compute_serial_position_curve` | Mean recall by within-list position, per hypothesis | trial-level `df` | DataFrame: hypothesis, within_list_pos, recall |
| `compute_recall_by_train` | Mean recall per train, per subject × hypothesis | trial-level `df` | DataFrame: hypothesis, subject_id, train_id, train_recall |
| `compute_drift_summary` | Mean encoding drift by boundary flag; compute drift contrast | trial-level `df` | Wide DataFrame: mean_nonboundary_drift, mean_boundary_drift, drift_boundary_contrast |
| `compute_rpe_recall_coupling` | Per-subject correlation between \|RPE\| and recall | trial-level `df` | DataFrame: hypothesis, subject_id, rpe_recall_r |
| `compute_subject_summary` | Merge all subject-level metrics into one wide table | trial-level `df` | One row per subject × hypothesis |
| `compute_hypothesis_summary` | Mean and SEM across subjects, per hypothesis | subject-level `df` | One row per hypothesis |
| `summarize_all_metrics` | One-call convenience returning all four metric outputs | trial-level `df` | Tuple of four DataFrames |

# `metrics.py` pseudocodes

## `compute_recall_accuracy`

**Implementation span:** `L20–L27`

### Pseudocode
- Group the trial DataFrame by hypothesis and subject_id.
- Compute the mean of the `recalled` column within each group.
- Rename the result column to `overall_recall` and return.

| Pseudocode step | Lines | Code |
|---|---|---|
| Group and compute mean | `L22–L24` | `df.groupby(["hypothesis", "subject_id"])["recalled"].mean()` |
| Reset index and rename | `L25–L27` | `.reset_index().rename(columns={"recalled": "overall_recall"})` |

## `compute_boundary_vs_nonboundary_recall`

**Implementation span:** `L30–L47`

### Pseudocode
- Group by hypothesis, subject_id, and is_boundary, then compute mean recall.
- Pivot the boundary flag to columns so each subject has one boundary and one non-boundary recall value.
- Rename the pivoted columns from `0` / `1` to `nonboundary_recall` / `boundary_recall`.
- Compute `boundary_advantage` as the difference.

| Pseudocode step | Lines | Code |
|---|---|---|
| Group and compute mean recall | `L32–L37` | `df.groupby([…, "is_boundary"])["recalled"].mean().reset_index()` |
| Pivot to wide form | `L39–L43` | `out.pivot_table(index=[…], columns="is_boundary", values="recall")` |
| Rename columns | `L44–L45` | `wide.rename(columns={0: "nonboundary_recall", 1: "boundary_recall"})` |
| Compute advantage | `L46` | `wide["boundary_advantage"] = wide["boundary_recall"] - wide["nonboundary_recall"]` |

## `compute_serial_position_curve`

**Implementation span:** `L50–L57`

### Pseudocode
- Group by hypothesis and within_list_pos, then compute mean recall.
- Return the result as a tidy DataFrame.

| Pseudocode step | Lines | Code |
|---|---|---|
| Group and mean | `L52–L54` | `df.groupby(["hypothesis", "within_list_pos"])["recalled"].mean()` |
| Reset index and rename | `L55–L57` | `.reset_index().rename(columns={"recalled": "recall"})` |

## `compute_recall_by_train`

**Implementation span:** `L60–L67`

### Pseudocode
- Group by hypothesis, subject_id, and train_id, then compute mean recall per train.
- Rename the output column to `train_recall` and return.

| Pseudocode step | Lines | Code |
|---|---|---|
| Group and mean | `L62–L64` | `df.groupby(["hypothesis", "subject_id", "train_id"])["recalled"].mean()` |
| Reset index and rename | `L65–L67` | `.reset_index().rename(columns={"recalled": "train_recall"})` |

## `compute_drift_summary`

**Implementation span:** `L70–L90`

### Pseudocode
- Group by hypothesis, subject_id, and is_boundary, then compute mean encoding drift.
- Pivot the boundary flag to columns so each subject has one boundary and one non-boundary drift value.
- Rename the pivoted columns to `mean_nonboundary_drift` and `mean_boundary_drift`.
- Compute `drift_boundary_contrast` as the difference.

| Pseudocode step | Lines | Code |
|---|---|---|
| Group and mean | `L72–L77` | `df.groupby([…, "is_boundary"])["encoding_drift"].mean().reset_index()` |
| Pivot to wide form | `L78–L82` | `out.pivot_table(index=[…], columns="is_boundary", values="mean_drift")` |
| Rename columns | `L83–L86` | `wide.rename(columns={0: "mean_nonboundary_drift", 1: "mean_boundary_drift"})` |
| Compute contrast | `L87–L89` | `wide["drift_boundary_contrast"] = wide["mean_boundary_drift"] - wide["mean_nonboundary_drift"]` |

## `compute_rpe_recall_coupling`

**Implementation span:** `L93–L105`

### Pseudocode
- Define a helper function that computes the Pearson correlation between \|RPE\| and recalled within a group, returning `NaN` if variance is zero.
- Group by hypothesis and subject_id, apply the helper.
- Rename the output column to `rpe_recall_r` and return.

| Pseudocode step | Lines | Code |
|---|---|---|
| Define inner correlation helper | `L95–L98` | `def _corr(g): … return g["abs_outcome_rpe"].corr(g["recalled"])` |
| Group and apply | `L100–L102` | `df.groupby(["hypothesis", "subject_id"]).apply(_corr, include_groups=False)` |
| Reset index and rename | `L103–L105` | `.reset_index().rename(columns={0: "rpe_recall_r"})` |

## `compute_subject_summary`

**Implementation span:** `L112–L138`

### Pseudocode
- Call each of the five Trial → Subject aggregation functions to get per-subject recall accuracy, boundary vs non-boundary recall, drift summary, and RPE-recall coupling.
- Additionally compute per-subject mean \|RPE\| and per-subject mean overall drift as separate aggregations.
- Left-merge all six result DataFrames on `(hypothesis, subject_id)`.
- Return the merged wide DataFrame.

| Pseudocode step | Lines | Code |
|---|---|---|
| Call component functions | `L114–L117` | `acc = compute_recall_accuracy(df); bnd = …; dft = …; rrc = …` |
| Compute mean \|RPE\| per subject | `L120–L125` | `mean_rpe = df.groupby(…)["abs_outcome_rpe"].mean().…` |
| Compute mean drift per subject | `L128–L133` | `mean_drift_all = df.groupby(…)["encoding_drift"].mean().…` |
| Left-merge all | `L135–L137` | `for right in [bnd, dft, mean_rpe, mean_drift_all, rrc]: out = out.merge(right, …)` |
| Return | `L138` | `return out` |

## `compute_hypothesis_summary`

**Implementation span:** `L145–L154`

### Pseudocode
- Identify all numeric columns in the subject-level DataFrame, excluding `subject_id`.
- Group by hypothesis and aggregate each numeric column with both `mean` and `sem`.
- Flatten the resulting multi-level column names into `metric_mean` / `metric_sem` format.
- Return the hypothesis-level summary.

| Pseudocode step | Lines | Code |
|---|---|---|
| Identify numeric columns | `L147–L149` | `numeric_cols = subject_df.select_dtypes(include="number").columns.tolist(); … remove subject_id` |
| Group and aggregate | `L151` | `agg = subject_df.groupby("hypothesis")[numeric_cols].agg(["mean", "sem"])` |
| Flatten column names | `L153` | `agg.columns = ["_".join(col) for col in agg.columns]` |
| Return | `L154` | `return agg.reset_index()` |

## `summarize_all_metrics`

**Implementation span:** `L161–L167`

### Pseudocode
- Call `compute_subject_summary` to get the full subject-level table.
- Call `compute_hypothesis_summary` on that table for aggregate statistics.
- Call `compute_serial_position_curve` for the SPC.
- Call `compute_recall_by_train` for train-level recall.
- Return all four outputs as a tuple.

| Pseudocode step | Lines | Code |
|---|---|---|
| Subject summary | `L163` | `subj = compute_subject_summary(trial_df)` |
| Hypothesis summary | `L164` | `hyp = compute_hypothesis_summary(subj)` |
| Serial position curve | `L165` | `spc = compute_serial_position_curve(trial_df)` |
| Train recall | `L166` | `trn = compute_recall_by_train(trial_df)` |
| Return tuple | `L167` | `return subj, hyp, spc, trn` |

# `plotting.py`

## Flowchart

```text
plotting.py
│
├── Module-level constants
│   ├── _PALETTE      hypothesis → colour hex                L19
│   ├── _HYP_ORDER    canonical plot ordering                L20
│   └── _HYP_LABELS   display-friendly names                 L21
│
├── Internal helpers
│   ├── _style()                                             L24–L26
│   │       └── set seaborn theme and rcParams
│   └── _hyp_label(name)                                     L29–L30
│           └── look up display label
│
├── Individual plots  (each: _style → build figure → return fig)
│   │
│   ├── plot_overall_recall(subject_df)                      L37–L52
│   ├── plot_boundary_recall(subject_df)                     L55–L78
│   ├── plot_boundary_advantage(subject_df)                  L81–L96
│   ├── plot_serial_position(spc_df)                         L99–L112
│   ├── plot_drift_distributions(trial_df)                   L115–L132
│   ├── plot_rpe_distributions(trial_df)                     L135–L147
│   ├── plot_recall_by_train(train_df)                       L150–L168
│   └── plot_mean_drift_by_item_type(subject_df)             L171–L193
│
├── Compact panel
│   │
│   └── make_compact_comparison_panel(subject_df, spc_df, train_df, trial_df)   L200–L306
│           └── 2×3 grid:
│               A. overall recall       B. boundary vs non-boundary
│               C. boundary advantage   D. drift by item type
│               E. serial position      F. recall by train
│
└── Save helper
    └── save_all_figures(figures, out_dir)                    L313–L322
            └── loop over {name: fig} dict, savefig as PNG
```

## Function summary

| Function | Purpose | Key inputs | Output |
|---|---|---|---|
| `_style` | Apply seaborn whitegrid theme and DPI settings | — | side-effect |
| `_hyp_label` | Map hypothesis key to display label | `name` | `str` |
| `plot_overall_recall` | Bar plot of mean overall recall ± SE | `subject_df` | `Figure` |
| `plot_boundary_recall` | Grouped bar: boundary vs non-boundary recall | `subject_df` | `Figure` |
| `plot_boundary_advantage` | Bar plot of boundary advantage | `subject_df` | `Figure` |
| `plot_serial_position` | Line plot of serial-position curves | `spc_df` | `Figure` |
| `plot_drift_distributions` | Box plots of encoding drift by boundary flag | `trial_df` | `Figure` |
| `plot_rpe_distributions` | KDE of \|RPE\| by hypothesis (sanity check) | `trial_df` | `Figure` |
| `plot_recall_by_train` | Line plot of mean within-train recall | `train_df` | `Figure` |
| `plot_mean_drift_by_item_type` | Grouped bar: drift by item type and hypothesis | `subject_df` | `Figure` |
| `make_compact_comparison_panel` | 2×3 grid of the six key diagnostic views | `subject_df`, `spc_df`, `train_df`, `trial_df` | `Figure` |
| `save_all_figures` | Save a dict of figures to PNG | `figures` dict, `out_dir` | side-effect: writes PNGs |

# `plotting.py` pseudocodes

## `_style`

**Implementation span:** `L24–L26`

### Pseudocode
- Set the seaborn theme to whitegrid with font_scale 1.1.
- Override figure and save DPI via rcParams.

| Pseudocode step | Lines | Code |
|---|---|---|
| Set theme | `L25` | `sns.set_theme(style="whitegrid", font_scale=1.1)` |
| Set DPI | `L26` | `plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, …})` |

## `_hyp_label`

**Implementation span:** `L29–L30`

### Pseudocode
- Look up the hypothesis name in the display-label dictionary; return the name unchanged if not found.

| Pseudocode step | Lines | Code |
|---|---|---|
| Dictionary lookup with fallback | `L30` | `return _HYP_LABELS.get(name, name)` |

## `plot_overall_recall`

**Implementation span:** `L37–L52`

### Pseudocode
- Apply house style.
- Create a bar plot of `overall_recall` grouped by hypothesis, with SE error bars.
- Set y-axis to [0, 1], relabel x-ticks, and return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Apply style, create figure | `L39–L40` | `_style(); fig, ax = plt.subplots(…)` |
| Draw bar plot | `L41–L45` | `sns.barplot(data=subject_df, x="hypothesis", y="overall_recall", …)` |
| Format axes | `L46–L50` | `ax.set_xticklabels(…); ax.set_ylabel(…); ax.set_ylim(0, 1)` |
| Return | `L52` | `return fig` |

## `plot_boundary_recall`

**Implementation span:** `L55–L78`

### Pseudocode
- Apply house style.
- Melt the subject DataFrame so `boundary_recall` and `nonboundary_recall` become rows with an `item_type` column.
- Map item_type values to display labels.
- Create a grouped bar plot with hypothesis on x and item_type as hue.
- Return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Melt to long form | `L58–L65` | `melted = subject_df.melt(…, value_vars=["boundary_recall", "nonboundary_recall"], …)` |
| Draw grouped bar | `L67–L70` | `sns.barplot(data=melted, x="hypothesis", y="recall", hue="item_type", …)` |
| Format and return | `L71–L78` | `ax.set_xticklabels(…); …; return fig` |

## `plot_boundary_advantage`

**Implementation span:** `L81–L96`

### Pseudocode
- Apply house style.
- Create a bar plot of `boundary_advantage` by hypothesis, with SE error bars.
- Add a dashed horizontal line at zero.
- Return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Draw bar plot | `L85–L89` | `sns.barplot(data=subject_df, x="hypothesis", y="boundary_advantage", …)` |
| Add zero line | `L94` | `ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")` |
| Return | `L96` | `return fig` |

## `plot_serial_position`

**Implementation span:** `L99–L112`

### Pseudocode
- Apply house style and create one axis.
- For each hypothesis, subset the SPC DataFrame and plot `within_list_pos` vs `recall` as a line.
- Add legend and return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Loop over hypotheses | `L103–L106` | `for hyp in _HYP_ORDER: sub = …; ax.plot(…)` |
| Return | `L112` | `return fig` |

## `plot_drift_distributions`

**Implementation span:** `L115–L132`

### Pseudocode
- Apply house style and copy the trial DataFrame.
- Create an `Item Type` column by mapping `is_boundary` to display labels.
- Draw box plots of `encoding_drift` with hypothesis on x and item type as hue, suppressing outliers.
- Return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Map boundary flag to label | `L119` | `trial_df["Item Type"] = trial_df["is_boundary"].map({0: "Non-boundary", 1: "Boundary"})` |
| Draw box plot | `L121–L125` | `sns.boxplot(data=trial_df, x="hypothesis", y="encoding_drift", hue="Item Type", …)` |
| Return | `L132` | `return fig` |

## `plot_rpe_distributions`

**Implementation span:** `L135–L147`

### Pseudocode
- Apply house style and create one axis.
- For each hypothesis, subset the trial DataFrame and plot a KDE of `abs_outcome_rpe`.
- Return the figure. (This plot is a sanity check: distributions should overlap across hypotheses.)

| Pseudocode step | Lines | Code |
|---|---|---|
| Loop and KDE | `L139–L142` | `for hyp in _HYP_ORDER: sub = …; sns.kdeplot(sub["abs_outcome_rpe"], …)` |
| Return | `L147` | `return fig` |

## `plot_recall_by_train`

**Implementation span:** `L150–L168`

### Pseudocode
- Apply house style.
- Aggregate `train_recall` by hypothesis and train_id (mean across subjects).
- For each hypothesis, plot train_id vs mean train recall as a line.
- Return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Aggregate across subjects | `L153–L157` | `agg = train_df.groupby(["hypothesis", "train_id"])["train_recall"].mean().reset_index()` |
| Loop and plot | `L159–L162` | `for hyp in _HYP_ORDER: sub = …; ax.plot(…)` |
| Return | `L168` | `return fig` |

## `plot_mean_drift_by_item_type`

**Implementation span:** `L171–L193`

### Pseudocode
- Apply house style.
- Melt the subject DataFrame so `mean_boundary_drift` and `mean_nonboundary_drift` become rows with an `item_type` column.
- Map item_type values to display labels.
- Draw a grouped bar plot with hypothesis on x and item_type as hue.
- Return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Melt to long form | `L174–L181` | `melted = subject_df.melt(…, value_vars=["mean_boundary_drift", "mean_nonboundary_drift"], …)` |
| Draw grouped bar | `L183–L186` | `sns.barplot(data=melted, x="hypothesis", y="drift", hue="item_type", …)` |
| Return | `L193` | `return fig` |

## `make_compact_comparison_panel`

**Implementation span:** `L200–L306`

### Pseudocode
- Apply house style and create a 2×3 subplot grid.
- **Panel A (0,0):** bar plot of overall recall by hypothesis.
- **Panel B (0,1):** grouped bar of boundary vs non-boundary recall (melt subject_df, map labels, barplot with hue).
- **Panel C (0,2):** bar plot of boundary advantage with a zero reference line.
- **Panel D (1,0):** grouped bar of mean encoding drift by item type (melt, map, barplot).
- **Panel E (1,1):** line plot of serial-position curves by hypothesis.
- **Panel F (1,2):** aggregate train-level recall across subjects, then line plot by hypothesis.
- Set suptitle, tight layout, and return the figure.

| Pseudocode step | Lines | Code |
|---|---|---|
| Create 2×3 grid | `L207–L208` | `_style(); fig, axes = plt.subplots(2, 3, figsize=(16, 9))` |
| Panel A: overall recall | `L211–L221` | `ax = axes[0,0]; sns.barplot(…, y="overall_recall", …)` |
| Panel B: boundary vs non-boundary | `L224–L242` | `ax = axes[0,1]; melted = …; sns.barplot(…, hue="item_type", …)` |
| Panel C: boundary advantage | `L245–L255` | `ax = axes[0,2]; sns.barplot(…, y="boundary_advantage", …); ax.axhline(0, …)` |
| Panel D: drift by item type | `L258–L275` | `ax = axes[1,0]; drift_m = …; sns.barplot(…, hue="item_type", …)` |
| Panel E: serial position | `L278–L286` | `ax = axes[1,1]; for hyp in …: ax.plot(…)` |
| Panel F: recall by train | `L289–L302` | `ax = axes[1,2]; train_agg = …; for hyp in …: ax.plot(…)` |
| Suptitle and layout | `L304–L306` | `fig.suptitle(…); fig.tight_layout(); return fig` |

## `save_all_figures`

**Implementation span:** `L313–L322`

### Pseudocode
- Convert the output directory to a `Path` and create it if needed.
- Loop over the `{name: fig}` dictionary and save each figure as a PNG.
- Print a confirmation with the count.

| Pseudocode step | Lines | Code |
|---|---|---|
| Ensure output dir | `L318–L319` | `out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)` |
| Loop and save | `L320–L321` | `for name, fig in figures.items(): fig.savefig(out_dir / f"{name}.png")` |
| Print confirmation | `L322` | `print(f"  Saved {len(figures)} figures to {out_dir}")` |